In [157]:
import pandas as pd
import numpy as np
import torch.nn as nn

In [158]:
df=pd.read_csv("powerplant_data.csv")
              

In [159]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [160]:
#At => temperture
#v => vacuum
#Ap => pressure
#RH=> humidity

#PE => produced energy


In [161]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [162]:
X=df.drop("PE",axis=1)
y=df["PE"]

In [163]:
#TRAIN TEST SPLIT

from sklearn.model_selection import train_test_split

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [164]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [165]:
#convert the data into tensors (pytorch core DS)

In [166]:
import torch
import torch.nn as nn

X_train_tensor =torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor =torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)


X_test_tensor =torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor =torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)




In [167]:
from torch.utils.data import TensorDataset , DataLoader

train_dataset =TensorDataset(X_train_tensor,y_train_tensor) 
test_dataset =TensorDataset(X_test_tensor,y_test_tensor ) 

In [168]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [169]:
# define the odelbulid the ANN model

import torch
import torch.nn as nn

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
          # 1st hidden layer
            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6,6),
            nn.ReLU(),

            # output layer
            nn.Linear(6,1)  
    )

    def forward(self,x):
       return self.model(x)

In [170]:
import torch.optim as optim

model=ANN()

#loss ,optimimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train the ANN
train_losses=[]
val_losses=[]
epochs =100

best_val_loss=float("inf")

for epoch in range(epochs):
   model.train()
   running_loss=0.0 #total training loss for one epoch
    
    
   for xb , yb in train_loader:
       #xb = fetures of 1 batch 
       #yb = labels of 1 batch
       optimizer.zero_grad()
       output= model(xb)#forward prop...predicted the output for this batch
       loss =crietrion(output,yb)
       loss.backward() # back prop...compute graident
       optimizer.step() #params update

       running_loss+= loss.item() #loss is a tensor ==> py float

   epoch_train_loss =running_loss / len(train_loader)
   train_losses.append(epoch_train_loss)


   #validation
   model.eval()
   running_val_loss=0.0


   with torch.no_grad():#neural network is already train so we dont require the grident update
                       #no gradient compute
     for xb , yb in test_loader:
       #xb = fetures of 1 batch 
       #yb = labels of 1 batch
       
        outputs= model(xb)#forward prop...predicted the output for this batch
        loss =crietrion(outputs,yb)
        running_val_loss=+loss 
         
   epoch_val_loss =running_val_loss / len(test_loader)
   val_losses.append( epoch_val_loss )

   print(f"epoch {epoch+1}/{epochs} ===> training loss= {epoch_train_loss} & val loss = {epoch_val_loss}")

   if epoch_val_loss < best_val_loss:
      best_val_loss=epoch_val_loss
      torch.save(model.state_dict(),"best_model.pt")

epoch 1/100 ===> training loss= 206392.335546875 & val loss = 3388.8017578125
epoch 2/100 ===> training loss= 195497.316796875 & val loss = 3028.48779296875
epoch 3/100 ===> training loss= 156068.45048828126 & val loss = 2217.07958984375
epoch 4/100 ===> training loss= 106211.72994791667 & val loss = 1549.611328125
epoch 5/100 ===> training loss= 74864.81248372396 & val loss = 1114.9178466796875
epoch 6/100 ===> training loss= 51162.46717122396 & val loss = 699.8639526367188
epoch 7/100 ===> training loss= 29615.716451009113 & val loss = 345.06927490234375
epoch 8/100 ===> training loss= 13928.571083577473 & val loss = 132.2617645263672
epoch 9/100 ===> training loss= 6579.1462809244795 & val loss = 63.77964401245117
epoch 10/100 ===> training loss= 4244.34443359375 & val loss = 42.30292510986328
epoch 11/100 ===> training loss= 3012.725397745768 & val loss = 29.55447769165039
epoch 12/100 ===> training loss= 2222.4250275929767 & val loss = 22.412487030029297
epoch 13/100 ===> training

In [ ]:
import matplotlib.pyplot as plt

loss_df= pd.DataFrame({
    "training loss":train_losses,
    "validation loss":val_losses
})
plt.plot(loss_df["training loss"],label="Training Loss")
plt.plot(loss_df["validation loss"],label="validation loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [ ]:
#loading the best model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
#evalution model